# Exercise 2: Merging, Aggregating, Filtering, and Visualizing

In [42]:
import altair as alt
import pandas as pd
from calitp_data_analysis.sql import to_snakecase

In [43]:
pd.options.display.max_columns = 100
pd.options.display.float_format = "{:.2f}".format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

* Read back in the `parquet` file with the `overall_score` you created from exercise 1.
* Read the Excel sheet containing the project information (scope of work, district, and project name).
* **Use f-strings.**

In [44]:
GCS_FILE_PATH = "gs://calitp-analytics-data/data-analyses/starter_kit/"
OVERALL_SCORE_NAME = "overall_score_parquet.parquet"
PROJECT_INFO_NAME = "starter_kit_csis_scoring_workbook.xlsx"

In [45]:
overall_score = pd.read_parquet(f"{GCS_FILE_PATH}{OVERALL_SCORE_NAME}")
project_info = to_snakecase(pd.read_excel(f"{GCS_FILE_PATH}{PROJECT_INFO_NAME}"))

In [46]:
overall_score.head()

,project_name,accessibility_score,dac_accessibility_score,dac_traffic_impacts_score,freight_efficiency_score,freight_sustainability_score,mode_shift_score,lu_natural_resources_score,safety_score,vmt_score,zev_score,public_engagement_score,climate_resilience_score,program_fit_score,overall_score
0,Meadow Magic Multi-Use Path,2,8,8,10,2,3,5,3,2,7,6,6,10,72
1,Bunny Hop Bike Boulevard,3,9,7,6,7,6,3,2,2,10,2,6,5,68
2,Strawberry Shortcake Sidewalks,7,8,7,10,8,9,5,6,2,10,2,8,5,87
3,River Ramble Rabbit Trail,4,8,6,10,2,10,3,2,5,4,8,8,5,75
4,Lilac Lane Dream Complete Street,6,1,10,7,5,2,6,6,9,3,7,4,6,72


In [47]:
project_info.head()

,ct_district,project_name,scope_of_work,project_cost,lead_agency
0,1,Meadow Magic Multi-Use Path,"A 2-mile Class I bike lane and multi-use path through a scenic meadow, featuring wildflower plantings, public art installations, and educational signage highlighting local wildlife.",5245734,Meadow Bunny Public Transportation (MBPT)
1,4,Bunny Hop Bike Boulevard,"A Class II bike lane with charming streetlights, benches, and bike racks designed to resemble carrot sticks, connecting residential neighborhoods to local schools and parks.",6929368,Unicorn Fairy Express Bus (UFX)
2,3,Strawberry Shortcake Sidewalks,"Colorful, patterned sidewalks connecting local schools and parks, incorporating playful strawberry-themed crosswalks and decorative street furniture.",4699350,Rainbow Mushroom Transportation Corporation (RMTC)
3,9,River Ramble Rabbit Trail,"A 5-mile Class III bike lane along a picturesque riverfront, offering stunning views, river access points, and interpretive signage sharing the area's natural and cultural history.",4800838,Strawberry Rainbow Transit Systems (SRTS)
4,6,Lilac Lane Dream Complete Street,"A vibrant Complete Street featuring bike lanes, wide sidewalks, and ample green space, prioritizing pedestrian safety and community engagement through public events and programming.",2398300,Fairy Creek Public Transit (FCPT)


## Merging 
* **Goal**: Your manager asks you to aggregate the dataframe by the Caltrans District grain to find
    * Median overall score
    * Max overall score 
    * Min overall score
    * Number of unique projects
* Annoyingly enough, the `overall_score` column and the `ct_district` are in two different dataframes. 
* You'll have to <b>merge</b> the dataframes on the common column(s) the two dataframes share.
* Welcome to DDS! This will happen to you all the time starting now. 

### Relevant Resources
* Read about and practice merges before continuing on the exercise. 
    * [Resource #1 is a great tutorial for beginners](https://www.practicalpythonfordatascience.com/03_cleaning_data.html?highlight=merge#merging-dataframes-together).
    * [Resource #2 is written by our own Tiffany Ku, but it contains some geospatial references so it's a bit more to digest](https://docs.calitp.org/data-infra/analytics_new_analysts/01-data-analysis-intro.html#merge-tabular-and-geospatial-data-for-data-analysis).
    

In [48]:
# Practice Here
import numpy as np
dim_animals = pd.DataFrame({
    "name": ["otters", "wasps", "ground squirrels", "bears", "scp-745"],
    "vertebrate": [False, False, True, True, True],
    "evil": [False, True, False, False, np.nan],
    "habitat": ["ocean", "woods", "field", "woods", "site 17"],
}).set_index("name")
dim_locations = pd.DataFrame({
    "name": ["monterey bay", "ucsc", "lake tahoe"],
    "population": [0, 30000, 10000],
}).set_index("name")
fct_place_habitat = pd.DataFrame({
    "location_name": ["monterey bay", "ucsc", "ucsc", "lake tahoe", "lake tahoe"],
    "habitat": ["ocean", "woods", "field", "woods", "ocean"],
})

locations_habitat_merged = dim_locations.merge(
    fct_place_habitat,
    how="outer",
    left_index=True,
    right_on="location_name",
    validate="one_to_many"
)
locations_with_animals = locations_habitat_merged.merge(
    dim_animals.reset_index(names="animal_name"),
    how="left",
    on="habitat",
    validate="many_to_many"
)
locations_with_animals

,population,location_name,habitat,animal_name,vertebrate,evil
0,0,monterey bay,ocean,otters,False,False
1,30000,ucsc,woods,wasps,False,True
2,30000,ucsc,woods,bears,True,False
3,30000,ucsc,field,ground squirrels,True,False
4,10000,lake tahoe,woods,wasps,False,True
5,10000,lake tahoe,woods,bears,True,False
6,10000,lake tahoe,ocean,otters,False,False


### Now merge your two CSIS dataframes
**Food for Thought**
* Which columns do the two dataframes have in common?
* What type of merge will achieve my goal?
    * Inner, outer, left, or right?
* What do I expect out of the merge?
    * Do I expect all the values of the merge keys to be 1:1? Or m:1? 
    * Do I expect a project to correspond with multiple districts? Maybe, projects can and do cross multiple boundaries.
    * Do I expect a project to correspond with only one total cost estimate value? Yes, there shouldn't be multiple cost estimates for the same project!
* How do I go about checking the data after the merge?
    * Which arguments are available to help me per the [docs](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.merge.html)?

### Double Checking
* How many rows do you expect?
* How many unique projects are there? 
* <b>Hint</b>: check the lengths of your original dataframes as well

### The Beauty of Outer Joins 
* As you have noticed, we are missing a couple of projects.
* This is where `outer` joins are very useful.
* Merge your dataframes again using an `outer` join and with `indicator = True` on.
    * `m2 = pd.merge(df1, df2, on=[column], indicator=True, how="outer")`
* Using `.value_counts()` on the column named `_merge` created by `indicator=True` to check out how many rows are found in both dataframes, the left only, and the right only

In [49]:
scores_districts_merged = overall_score.merge(
    project_info[["ct_district", "project_name"]],
    on="project_name",
    how="outer",
    indicator=True,
    validate="one_to_one"
)
scores_districts_merged._merge.value_counts()

both          41
left_only      3
right_only     3
Name: _merge, dtype: int64

### Filtering
* Filter out for only the `left_only` and `right_only` values.
    * `!=` means does not equal to. 
    * `==` means equal to.

In [50]:
scores_districts_merged.loc[scores_districts_merged._merge.isin(["left_only", "right_only"])]

,project_name,accessibility_score,dac_accessibility_score,dac_traffic_impacts_score,freight_efficiency_score,freight_sustainability_score,mode_shift_score,lu_natural_resources_score,safety_score,vmt_score,zev_score,public_engagement_score,climate_resilience_score,program_fit_score,overall_score,ct_district,_merge
10,Rainbow Rush HOT Lanes,2.00,6.00,7.00,6.00,2.00,3.00,8.00,5.00,10.00,3.00,9.00,5.00,7.00,73.00,NaN,left_only
12,Bunny Lane HOV+2 Haven,10.00,1.00,10.00,9.00,9.00,8.00,8.00,4.00,3.00,8.00,8.00,6.00,1.00,85.00,NaN,left_only
26,Main Street Muffin Top Revitalization,9.00,9.00,5.00,8.00,7.00,10.00,6.00,9.00,9.00,7.00,1.00,10.00,7.00,97.00,NaN,left_only
44,Rainbow Rush hot Lanes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.00,right_only
45,Bunny Lane HOV+2 heaven,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.00,right_only
46,main street muffin top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.00,right_only


* You could also use `isin([list of elements you want to keep])` to retain multiple elements you want.

In [51]:
#see above

* If you want to filter out multiple elements use `~df.column.isin([list of elements you don't want to keep])`

In [52]:
scores_districts_merged.loc[~scores_districts_merged._merge.isin(["left_only", "right_only"])].head()

,project_name,accessibility_score,dac_accessibility_score,dac_traffic_impacts_score,freight_efficiency_score,freight_sustainability_score,mode_shift_score,lu_natural_resources_score,safety_score,vmt_score,zev_score,public_engagement_score,climate_resilience_score,program_fit_score,overall_score,ct_district,_merge
0,Meadow Magic Multi-Use Path,2.00,8.00,8.00,10.00,2.00,3.00,5.00,3.00,2.00,7.00,6.00,6.00,10.00,72.00,1.00,both
1,Bunny Hop Bike Boulevard,3.00,9.00,7.00,6.00,7.00,6.00,3.00,2.00,2.00,10.00,2.00,6.00,5.00,68.00,4.00,both
2,Strawberry Shortcake Sidewalks,7.00,8.00,7.00,10.00,8.00,9.00,5.00,6.00,2.00,10.00,2.00,8.00,5.00,87.00,3.00,both
3,River Ramble Rabbit Trail,4.00,8.00,6.00,10.00,2.00,10.00,3.00,2.00,5.00,4.00,8.00,8.00,5.00,75.00,9.00,both
4,Lilac Lane Dream Complete Street,6.00,1.00,10.00,7.00,5.00,2.00,6.00,6.00,9.00,3.00,7.00,4.00,6.00,72.00,6.00,both


### Dictionaries
* String data is often entered in many different ways. 
    * BART can be entered in as bart, Bay Area Rapid Transit, BaRT, and more. 
* Often, differing strings between two dataframes are the reason why your dataframe is not merging properly.
* In Excel, it's easy to go in and manually tweak everything. However, that is not reproducible and time consuming. 
* Luckily with Python we can automate this. 
* Since there are only a couple of names to replace, we can do it using a <b>dictionary</b>.

#### What is a dictionary?
* Per Practical Python for Data Science, a dictionary is <i>Dictionaries are used to store data values in key:value pairs. Similar to the list, a dictionary is a collection of objects. It is also mutable, meaning that you can add, remove, change values inside of it...With the list, we access elements using the index. With the dictionary, we access elements using keys.</i>
* Dictionaries are very important.
* Read more [here](https://www.practicalpythonfordatascience.com/00_python_crash_course_datatypes.html?highlight=dictionary#dictionary) and **follow its example in the cells below.**
    

In [53]:
# Practice Here
population_nyc = {
    'bronx': 1472654,
    'brooklyn': 2736074,
    'manhattan': 1694251, 
    'queens': 2405464,
    'staten_island': 495747
}
print(f"Manhattan population {population_nyc['manhattan']}")
print(population_nyc.keys())
population_nyc["long_island"] = 80631232
population_nyc["long_island"] = 8
population_nyc.pop("long_island")

Manhattan population 1694251
dict_keys(['bronx', 'brooklyn', 'manhattan', 'queens', 'staten_island'])


8

#### Application of Dictionaries: Replacing Values
* [Resource](https://www.practicalpythonfordatascience.com/03_cleaning_data#recoding-column-values)
* **Step 1**: Filter out for the rows that <b>didn't</b> merge. Find the unique values of the `project_name` column using `.unique()`
* Take a look at elements using 
    * Trailing white spaces
    * Capitalization
    * Spelling
    * Symbols

In [54]:
unmerged_projects = scores_districts_merged.loc[scores_districts_merged._merge.isin(["left_only", "right_only"]), ["project_name", "_merge"]]
unmerged_projects

,project_name,_merge
10,Rainbow Rush HOT Lanes,left_only
12,Bunny Lane HOV+2 Haven,left_only
26,Main Street Muffin Top Revitalization,left_only
44,Rainbow Rush hot Lanes,right_only
45,Bunny Lane HOV+2 heaven,right_only
46,main street muffin top,right_only


In [55]:
unmerged_projects["project_name"].unique()

array(['Rainbow Rush HOT Lanes', 'Bunny Lane HOV+2 Haven',
       'Main Street Muffin Top Revitalization', 'Rainbow Rush hot  Lanes',
       'Bunny Lane HOV+2 heaven', 'main street muffin top '], dtype=object)

* **Step 2:** Decide whether you want to rename the values in the left dataframe or the right one. 
* **Step 3:** The <b>keys</b>, are the values you want to replace. The <b>values</b>, are what you want to replace these values with. 
    * Let's say my left value is "AC Transit" but I want it to be "Alameda Contra Costa County Transit", my dictionary would be 
    * `my_dict = {"AC Transit":"Alameda Contra Costa County Transit"}`

In [56]:
CORRECTIONS = {
    "Rainbow Rush hot  Lanes": "Rainbow Rush HOT Lanes",
    "Bunny Lane HOV+2 heaven": "Bunny Lane HOV+2 Haven",
    "main street muffin top ": "Main Street Muffin Top Revitalization",
}

* **Step 4**: Use your dictionary in `.replace()` to recode the values.

In [57]:
project_info.project_name = project_info.project_name.replace(CORRECTIONS)

#### Merge your dataframes again. 
* This time the number of unique project names should match the rows of the merged dataframe perfectly.
* Make sure to double check that!

In [58]:
project_info_with_corrections = project_info.copy()

scores_districts_merged_names_fixed = overall_score.merge(
    project_info,
    how="outer",
    on="project_name",
    validate="one_to_one",
)
assert scores_districts_merged_names_fixed.project_name.nunique() == project_info.project_name.nunique()
scores_districts_merged_names_fixed.head()

,project_name,accessibility_score,dac_accessibility_score,dac_traffic_impacts_score,freight_efficiency_score,freight_sustainability_score,mode_shift_score,lu_natural_resources_score,safety_score,vmt_score,zev_score,public_engagement_score,climate_resilience_score,program_fit_score,overall_score,ct_district,scope_of_work,project_cost,lead_agency
0,Meadow Magic Multi-Use Path,2,8,8,10,2,3,5,3,2,7,6,6,10,72,1,"A 2-mile Class I bike lane and multi-use path through a scenic meadow, featuring wildflower plantings, public art installations, and educational signage highlighting local wildlife.",5245734,Meadow Bunny Public Transportation (MBPT)
1,Bunny Hop Bike Boulevard,3,9,7,6,7,6,3,2,2,10,2,6,5,68,4,"A Class II bike lane with charming streetlights, benches, and bike racks designed to resemble carrot sticks, connecting residential neighborhoods to local schools and parks.",6929368,Unicorn Fairy Express Bus (UFX)
2,Strawberry Shortcake Sidewalks,7,8,7,10,8,9,5,6,2,10,2,8,5,87,3,"Colorful, patterned sidewalks connecting local schools and parks, incorporating playful strawberry-themed crosswalks and decorative street furniture.",4699350,Rainbow Mushroom Transportation Corporation (RMTC)
3,River Ramble Rabbit Trail,4,8,6,10,2,10,3,2,5,4,8,8,5,75,9,"A 5-mile Class III bike lane along a picturesque riverfront, offering stunning views, river access points, and interpretive signage sharing the area's natural and cultural history.",4800838,Strawberry Rainbow Transit Systems (SRTS)
4,Lilac Lane Dream Complete Street,6,1,10,7,5,2,6,6,9,3,7,4,6,72,6,"A vibrant Complete Street featuring bike lanes, wide sidewalks, and ample green space, prioritizing pedestrian safety and community engagement through public events and programming.",2398300,Fairy Creek Public Transit (FCPT)


#### Save this dataframe as a parquet to GCS under a new name
* Use a `f-string`

In [59]:
PROJECT_SCORES_BY_DISTRICT_NAME = "project_scores_by_district.parquet"
scores_districts_merged_names_fixed.to_parquet(f"{GCS_FILE_PATH}{PROJECT_SCORES_BY_DISTRICT_NAME}")

## Groupby
* You're done merging...Oh wait, that wasn't even part of your manager's request. You still need to aggregate. 
* By Caltrans District to find
    * Median overall score
    * Max overall score 
    * Min overall score
    * Number of unique projects
* There are many options Some are `groupby / agg`, `pivot_table`, `groupby / transform`
* <b>Resource</b>: 
    * [DDS Docs](https://docs.calitp.org/data-infra/analytics_new_analysts/01-data-analysis-intro.html#aggregating)

In [60]:
# Practice tutorial linked above here
locations_with_animals.groupby("location_name").agg({
    "evil": "sum",
    "animal_name": "count",
    "habitat": "nunique",
}).replace({False: 0}).reset_index()

,location_name,evil,animal_name,habitat
0,lake tahoe,1,3,2
1,monterey bay,0,1,1
2,ucsc,1,3,2


### Apply your new knowledge to the prompt above.
* Hint: After aggregating, some of the column names will no longer be relevant. 
* For example, if you use `scope_of_work` to count the number of projects, this column no longer represents `scope_of_work`.
* It should be renamed something like `n_projects`.
* Rename your columns using this `df.rename(columns={"old_column_name":"new_column_name"})`

In [61]:
district_summary = pd.DataFrame(index=project_info["ct_district"].unique()).sort_index()
score_district_group = scores_districts_merged_names_fixed.groupby("ct_district")
district_summary["n_projects"] = score_district_group["project_name"].nunique()
district_summary["max_overall_score"] = score_district_group["overall_score"].max()
district_summary["min_overall_score"] = score_district_group["overall_score"].min()
district_summary["mean_overall_score"] = score_district_group["overall_score"].mean()
district_summary["median_overall_score"] = score_district_group["overall_score"].median()
district_summary

,n_projects,max_overall_score,min_overall_score,mean_overall_score,median_overall_score
1,1,72,72,72.00,72.00
2,2,63,60,61.50,61.50
3,6,97,54,77.67,80.50
4,6,97,60,73.17,70.50
5,4,98,58,77.50,77.00
6,3,77,63,70.67,72.00
7,3,94,79,85.00,82.00
8,5,85,66,75.20,73.00
9,3,87,67,76.33,75.00
10,2,86,59,72.50,72.50


### Styling a Dataframe
* `pandas` has quite a few options that allow you to style your dataframe.
* [This tutorial](https://betterdatascience.com/style-pandas-dataframes/) offers some great ways to jazz up your dataframe. (dead link)
* You can always read the [pandas documentation](https://pandas.pydata.org/pandas-docs/stable/user_guide/style.html) for more ideas.
* Some ideas:
    * Change the font
    * Turn off the index
    * Use colors to code low-high values
    * Change the alignment of the values

In [62]:
# Practice here 
s = district_summary.style

s.set_table_styles([
    {
        "selector": "th:not(.index_name)",
        'props': 'font-style: bold; font-family: Impact;'
    },
    {
        "selector": ".true",
        "props": "background-color: #ff1030",
    },
    {
        "selector": ".false",
        "props": "background-color: #00ff00"
    }
]).set_td_classes((district_summary > district_summary.mean(axis=0)).replace({True: "true", False: "false"}))

,n_projects,max_overall_score,min_overall_score,mean_overall_score,median_overall_score
1,1,72,72,72.000000,72.000000
2,2,63,60,61.500000,61.500000
3,6,97,54,77.666667,80.500000
4,6,97,60,73.166667,70.500000
5,4,98,58,77.500000,77.000000
6,3,77,63,70.666667,72.000000
7,3,94,79,85.000000,82.000000
8,5,85,66,75.200000,73.000000
9,3,87,67,76.333333,75.000000
10,2,86,59,72.500000,72.500000


In [63]:
district_summary > district_summary.mean(axis=0)

,n_projects,max_overall_score,min_overall_score,mean_overall_score,median_overall_score
1,False,False,True,False,False
2,False,False,False,False,False
3,True,True,False,True,True
4,True,True,False,False,False
5,True,True,False,True,True
6,False,False,True,False,False
7,False,True,True,True,True
8,True,False,True,True,False
9,False,True,True,True,True
10,False,False,False,False,False


### Altair
* While a table is great, sometimes a chart is a better way to display an insight.
* Our preferred visualization library is `Altair`.
    * Docs page is [here](https://altair-viz.github.io/).
* The code to create a simple bar chart goes something like this. 
    * `alt.Chart(source).mark_bar().encode(x='a',y='b')`
    * `source` is the dataframe you want to use for your chart.
    * `x` denotes the column you are plotting on the X-axis. Make sure your column name has quotation marks around it. 
    * `y` denotes the column you are plotting on the Y-axis. 
* <b>Make your first chart below.</b>

In [64]:
district_summary_to_plot = district_summary.reset_index(names="ct_district")
alt.Chart(district_summary_to_plot).mark_bar().encode(
    x="ct_district", y="n_projects"
)

alt.Chart(...)

#### Customizing
* `altair` offers an endless ways to amp up the personality of your chart.
* Additionally, the chart above without a title and legend is a data visualization "taboo" and the dull blue is uninspiring. 

#### Add a title
* You can do so within  `.Chart()`
`alt.Chart(source,  title="your_title_here").mark_bar().encode(x='a',y='b')`

In [65]:
alt.Chart(district_summary_to_plot, title="# projects by ct district").mark_bar().encode(
    x="ct_district", y="n_projects"
)

alt.Chart(...)

#### Different Charts
* If you want something that isn't a bar chart, simply swap out `.mark_bar()` for `.mark_line` or `mark_circle`.


In [66]:
alt.Chart(district_summary_to_plot, title="# projects by district").mark_line().encode(
    x="ct_district", y="n_projects"
)

alt.Chart(...)

#### Add some color/DDS's Python Library
* We have some default color palettes that are already in our [internal library of functions](https://docs.calitp.org/data-infra/analytics_tools/python_libraries.html#calitp-data-analysis).

In [67]:
# Import the color palettes
from calitp_data_analysis import calitp_color_palette

* To see what is inside a module,  put two question marks behind it.
* From here, you can choose another color palette.

In [68]:
calitp_color_palette??

Type:        module
String form: <module 'calitp_data_analysis.calitp_color_palette' from '/opt/conda/lib/python3.11/site-packages/calitp_data_analysis/calitp_color_palette.py'>
File:        /opt/conda/lib/python3.11/site-packages/calitp_data_analysis/calitp_color_palette.py
Source:     
# --------------------------------------------------------------#
# Cal-ITP style guide
# Google Drive > Cal-ITP Team > Project Resources >
# Branded Resources and External Comms Guidelines > Branded Resources > Style Guide
# --------------------------------------------------------------#
CALITP_CATEGORY_BRIGHT_COLORS = [
    "#2EA8CE",  # darker blue
    "#EB9F3C",  # orange
    "#F4D837",  # yellow
    "#51BF9D",  # green
    "#8CBCCB",  # lighter blue
    "#9487C0",  # purple
]

CALITP_CATEGORY_BOLD_COLORS = [
    "#136C97",  # darker blue
    "#E16B26",  # orange
    "#F6BF16",  # yellow
    "#00896B",  # green
    "#7790A3",  # lighter blue
    "#5B559C",  # purple
]

CALITP_DIVERGING_COLORS = [
 

* Place the column you want the colors to be based on in `color=alt.Color(column)`
* Place your color palette in the `scale` argument `scale=alt.Scale(range=your_color_palette)`.

In [69]:
alt.Chart(district_summary_to_plot, title="# projects by ct district").mark_bar().encode(
    x="ct_district",
    y="n_projects",
    color=alt.Color(
        "n_projects",  # This is the column you want the color of your bar to be based on
        title="# projects",  # This is the legend of your title
        scale=alt.Scale(
            range=calitp_color_palette.CALITP_CATEGORY_BOLD_COLORS   # This is where you can customize the colors,
        ),
    ),
)

alt.Chart(...)

#### Adjusting the Axis
* Sometimes, we want to adjust the axis to have a min and max value.
* You do so using the `scale=alt.Scale(domain=[min_value, max_value]))` argument behind the X and Y axis.

In [70]:
alt.Chart(district_summary_to_plot, title="# projects by ct district").mark_bar().encode(
    x=alt.X("ct_district", scale=alt.Scale(domain=[1, 12])),
    y=alt.Y("n_projects", scale=alt.Scale(domain=[0, 10])),
    color=alt.Color(
        "n_projects",
        title="# projects",
        scale=alt.Scale(range=calitp_color_palette.CALITP_DIVERGING_COLORS, domain=[2,5]),
    ),
)

alt.Chart(...)

### Finishing Touches 
* `.properties(width=400, height=250)` adjusts the size of your chart. 
* `tooltip=[columns you want]` allows you to create a tooltip that pops up when you hover over each bar/circle/etc.
* `.mark_bar(size=10)` adjusts the size of the bar/circle/etc.

In [71]:
alt.Chart(district_summary_to_plot, title="# projects by district").mark_bar(size=10).encode(
    x=alt.X("ct_district", scale=alt.Scale(domain=[1, 12])),
    y=alt.Y("n_projects", scale=alt.Scale(domain=[0, 10])),
    color=alt.Color(
        "n_projects",
        title="# projects",
        scale=alt.Scale(range=calitp_color_palette.CALITP_DIVERGING_COLORS),
    ),
    tooltip=["ct_district", "n_projects"],
).properties(width=400, height=250)

alt.Chart(...)

### We have only visualized one column of data. 
* We have only visualized one column of data, but we have a couple of columns above. 
* Try to customize your graph. If you can dream it, you can probably do it with Altair. 
    * You can turn off the grid lines, rotate the axis labels by various degrees, label the bars, add a dropdown menu to change the axis, and more. 
* Make a few other charts in different styles.
* Inspiration
    * Altair's [gallery](https://altair-viz.github.io/gallery/index.html)
    * DDS's [portfolio](https://analysis.calitp.org/)


In [75]:
alt.Chart(
    district_summary_to_plot, title="Median Project Score by District",
).mark_bar(size=12).encode(
    x=alt.X("ct_district"),
    y=alt.Y("median_overall_score"),
    color=alt.Color(
        "median_overall_score",
        title="Median Overall Score",
        scale=alt.Scale(range=calitp_color_palette.CALITP_CATEGORY_BRIGHT_COLORS),
    )
)

alt.Chart(...)